In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
import os
import urllib3
import re  # 정규표현식 라이브러리 추가

# SSL 인증서 관련 오류 방지
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['CURL_CA_BUNDLE'] = ''

# 1. 서울 구 리스트
seoul_gu_list = [
    "강남구", "강동구", "강북구", "강서구", "관악구", "광진구", "구로구", "금천구",
    "노원구", "도봉구", "동대문구", "동작구", "마포구", "서대문구", "서초구", "성동구",
    "성북구", "송파구", "양천구", "영등포구", "용산구", "은평구", "종로구", "중구", "중랑구"
]

def collect_mega_coffee():
    all_stores = []
    base_url = "https://www.mega-mgccoffee.com/store/find/store_search.php"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"데이터 수집 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("-" * 50)

    for gu in seoul_gu_list:
        params = {"store_search": gu}
        try:
            response = requests.get(base_url, params=params, headers=headers, verify=False)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            li_list = soup.select('li > a > div.cont_text')
            
            for item in li_list:
                # 5. 매장명 파싱
                name_div = item.find('div', class_='cont_text_inner')
                store_name = name_div.find('b').get_text(strip=True) if name_div and name_div.find('b') else "N/A"
                
                # 6~7. 주소 및 전화번호 분리 로직 (02- 및 070- 대응)
                info_div = item.find('div', class_='cont_text_info')
                if info_div:
                    raw_text = info_div.get_text().strip()
                    
                    # 정규표현식으로 02- 또는 070- 이 시작되는 위치를 찾음
                    # r'(02-|070-)'는 "02-" 또는 "070-" 중 먼저 걸리는 것을 의미함
                    match = re.search(r'(02-|070-)', raw_text)
                    
                    if match:
                        split_idx = match.start() # 찾은 패턴이 시작되는 인덱스 번호
                        address = raw_text[:split_idx].strip()
                        phone = raw_text[split_idx:].strip()
                    else:
                        # 전화번호 패턴이 발견되지 않은 경우
                        address = raw_text
                        phone = ""

                    # 10. 주소가 "서울"로 시작하는 데이터만 포함
                    if address.startswith("서울"):
                        all_stores.append({
                            "매장명": store_name,
                            "주소": address,
                            "전화번호": phone
                        })
            
            # 8. 한 구가 끝날 때마다 시간 기록
            current_time = datetime.now().strftime('%H:%M:%S')
            print(f"[{current_time}] {gu} 완료 (현재 누적 매장 수: {len(all_stores)}개)")
            
            time.sleep(0.6)
            
        except Exception as e:
            print(f"{gu} 수집 중 에러 발생: {e}")

    # 10. 중복 데이터 제거
    df = pd.DataFrame(all_stores)
    df = df.drop_duplicates(subset=['매장명', '주소'], keep='first')

    # 9. CSV 저장
    df.to_csv("mega_coffee.csv", index=False, encoding="utf-8-sig")
    print("-" * 50)
    print(f"최종 수집 완료! 총 {len(df)}개 매장이 'mega_coffee.csv' 파일로 저장되었습니다.")

if __name__ == "__main__":
    collect_mega_coffee()

데이터 수집 시작: 2026-03-28 14:52:41
--------------------------------------------------
[14:52:41] 강남구 완료 (현재 누적 매장 수: 85개)
[14:52:42] 강동구 완료 (현재 누적 매장 수: 129개)
[14:52:43] 강북구 완료 (현재 누적 매장 수: 155개)
[14:52:44] 강서구 완료 (현재 누적 매장 수: 209개)
[14:52:44] 관악구 완료 (현재 누적 매장 수: 249개)
[14:52:45] 광진구 완료 (현재 누적 매장 수: 285개)
[14:52:46] 구로구 완료 (현재 누적 매장 수: 327개)
[14:52:47] 금천구 완료 (현재 누적 매장 수: 357개)
[14:52:47] 노원구 완료 (현재 누적 매장 수: 401개)
[14:52:48] 도봉구 완료 (현재 누적 매장 수: 428개)
[14:52:49] 동대문구 완료 (현재 누적 매장 수: 466개)
[14:52:50] 동작구 완료 (현재 누적 매장 수: 488개)
[14:52:50] 마포구 완료 (현재 누적 매장 수: 540개)
[14:52:51] 서대문구 완료 (현재 누적 매장 수: 568개)
[14:52:52] 서초구 완료 (현재 누적 매장 수: 616개)
[14:52:53] 성동구 완료 (현재 누적 매장 수: 648개)
[14:52:53] 성북구 완료 (현재 누적 매장 수: 689개)
[14:52:54] 송파구 완료 (현재 누적 매장 수: 749개)
[14:52:55] 양천구 완료 (현재 누적 매장 수: 790개)
[14:52:56] 영등포구 완료 (현재 누적 매장 수: 848개)
[14:52:57] 용산구 완료 (현재 누적 매장 수: 867개)
[14:52:58] 은평구 완료 (현재 누적 매장 수: 902개)
[14:52:59] 종로구 완료 (현재 누적 매장 수: 935개)
[14:53:00] 중구 완료 (현재 누적 매장 수: 972개)
[14:53:00] 중랑구 완료 (현재 누적 매장 수

In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
import os
import urllib3
import re

# SSL 인증서 및 경고 무시 설정
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['CURL_CA_BUNDLE'] = ''

# 1. 제주특별자치도 시 리스트 (2개 지역)
jeju_list = ["제주시", "서귀포시"]

def collect_mega_jeju():
    new_stores = []
    base_url = "https://www.mega-mgccoffee.com/store/find/store_search.php"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # 마스터 전화번호 패턴 (제주 064 포함 전지역 대응)
    master_phone_regex = r'(02-|03[1-3]-|04[1-4]-|05[1-5]-|06[1-4]-|070-|050\d?-|15\d{2}-|16\d{2}-|18\d{2}-|010-)'

    print(f"제주특별자치도 데이터 수집 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("-" * 50)

    for area in jeju_list:
        # 지역명만 사용하여 검색 (제주 제주시 X -> 제주시 O)
        params = {"store_search": area}
        try:
            response = requests.get(base_url, params=params, headers=headers, verify=False)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            li_list = soup.select('li > a > div.cont_text')
            
            for item in li_list:
                # 5. 매장명 파싱
                name_div = item.find('div', class_='cont_text_inner')
                store_name = name_div.find('b').get_text(strip=True) if name_div and name_div.find('b') else "N/A"
                
                # 6~7. 주소 및 전화번호 분리
                info_div = item.find('div', class_='cont_text_info')
                if info_div:
                    raw_text = info_div.get_text().strip()
                    
                    # 마스터 패턴으로 전화번호 시작 위치 찾기
                    match = re.search(master_phone_regex, raw_text)
                    
                    if match:
                        split_idx = match.start()
                        address = raw_text[:split_idx].strip()
                        phone = raw_text[split_idx:].strip()
                    else:
                        address = raw_text
                        phone = ""

                    # 10. 주소가 "제주"로 시작하는 데이터만 필터링
                    if address.startswith("제주"):
                        new_stores.append({
                            "매장명": store_name,
                            "주소": address,
                            "전화번호": phone
                        })
            
            # 8. 한 지역이 끝날 때마다 시간 기록
            current_time = datetime.now().strftime('%H:%M:%S')
            print(f"[{current_time}] {area} 완료 (누적 제주 매장: {len(new_stores)}개)")
            
            time.sleep(0.5)
            
        except Exception as e:
            print(f"{area} 수집 중 에러 발생: {e}")

    # --- 최종 파일 병합 및 중복 제거 ---
    file_name = "mega_coffee.csv"
    df_new = pd.DataFrame(new_stores)
    
    if os.path.exists(file_name):
        df_old = pd.read_csv(file_name)
        # 지금까지 모은 모든 지역 데이터와 병합
        df_final = pd.concat([df_old, df_new], ignore_index=True)
        print(f"\n기존 전국 데이터에 제주도 매장 {len(df_new)}개를 추가합니다.")
    else:
        df_final = df_new
        print(f"\n기존 파일이 없어 새로운 파일을 생성합니다.")

    # 10. 전체 데이터 중복 제거 (최종 점검)
    df_final = df_final.drop_duplicates(subset=['매장명', '주소'], keep='first')

    # 9. CSV로 최종 저장
    df_final.to_csv(file_name, index=False, encoding="utf-8-sig")
    print("-" * 50)
    print("🎉 축하합니다! 대한민국 전역의 메가커피 데이터 수집이 완료되었습니다.")
    print(f"최종 저장된 전국의 총 매장 수: {len(df_final)}개")
    print(f"파일 경로: {os.path.abspath(file_name)}")

if __name__ == "__main__":
    collect_mega_jeju()

제주특별자치도 데이터 수집 시작: 2026-03-28 15:14:17
--------------------------------------------------
[15:14:17] 제주시 완료 (누적 제주 매장: 33개)
[15:14:18] 서귀포시 완료 (누적 제주 매장: 48개)

기존 전국 데이터에 제주도 매장 48개를 추가합니다.
--------------------------------------------------
🎉 축하합니다! 대한민국 전역의 메가커피 데이터 수집이 완료되었습니다.
최종 저장된 전국의 총 매장 수: 4039개
파일 경로: c:\PYTHON\crawling_archive\mega_coffee.csv


In [27]:
import pandas as pd
import re

def preprocess_mega_data():
    file_name = "mega_coffee.csv"
    
    # 1. CSV 파일을 데이터프레임으로 변환
    try:
        df = pd.read_csv(file_name)
        print(f"데이터 로드 완료! 전체 매장 수: {len(df)}개")
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다. 경로를 확인해주세요: {file_name}")
        return

    # --- 전처리 및 문제 데이터 탐색 ---

    # 2. 전화번호 컬럼에 한글이 포함된 행들 찾아내기
    # [가-힣] 정규표현식을 사용하여 한글이 한 글자라도 들어있으면 추출합니다.
    # na=False 옵션으로 결측치(NaN)는 제외하고 검색합니다.
    has_hangul = df[df['전화번호'].str.contains('[가-힣]', na=False)]

    # 3. 전화번호가 없는(결측치 또는 빈 문자열) 행들 찾아내기
    # isna()로 NaN 값을 찾고, strip() 후 빈 문자열("")인지 확인합니다.
    is_empty = df[df['전화번호'].isna() | (df['전화번호'].str.strip() == "")]

    # --- 결과 리포트 출력 ---
    print("\n" + "="*50)
    print(f"🔍 [데이터 점검 결과 보고서]")
    print("="*50)

    print(f"1. 전화번호에 '한글'이 섞여 있는 데이터: {len(has_hangul)}건")
    if not has_hangul.empty:
        print("-" * 30)
        # 매장명과 문제의 전화번호만 출력해봅니다.
        print(has_hangul[['매장명', '전화번호']])
    
    print("\n" + "-" * 50)
    
    print(f"2. 전화번호가 '비어 있는' 데이터: {len(is_empty)}건")
    if not is_empty.empty:
        print("-" * 30)
        # 전화번호가 없는 매장들의 매장명을 출력해봅니다.
        print(is_empty[['매장명', '주소']])

    print("\n" + "="*50)
    
    # 필요하다면 이후에 이 데이터들을 어떻게 처리할지(삭제 혹은 수정) 결정할 수 있습니다.
    return df, has_hangul, is_empty

if __name__ == "__main__":
    df, hangul_rows, empty_rows = preprocess_mega_data()

데이터 로드 완료! 전체 매장 수: 4039개

🔍 [데이터 점검 결과 보고서]
1. 전화번호에 '한글'이 섞여 있는 데이터: 1건
------------------------------
           매장명                             전화번호
2999  대전유성온천역점  02-1호\t\t\t\t\t\t\t042-822-7054

--------------------------------------------------
2. 전화번호가 '비어 있는' 데이터: 24건
------------------------------
             매장명                                                 주소
254      건대학생회관점            서울특별시 광진구 화양동 393-1 , 1층\t\t\t\t\t\t\t-
301     구로디지털단지점  서울특별시 구로구 디지털로31길 41 , (구로동, 이앤씨벤처드림타워6차)110호 ...
368        노원중앙점                        서울특별시 노원구 노해로 509 , 1층(상계동)
593       서울서일초점                          서울 서초구 남부순환로339길 53 (서초동)
732         오금역점  서울특별시 송파구 오금로 306, 1층 113호(가락동 , 가락스타클래스)\t\t\...
738      잠실롯데월드점                        서울 송파구 올림픽로 240 (잠실동, 롯데월드)
852    신용산역지하상가점               서울 용산구 한강대로 지하 112 (한강로2가, 4호선 신용산역)
997      중랑면목시장점  서울 중랑구 면목로 284 (면목동, HY에코시티빌)\t\t\t\t\t\t\t010...
1149       가천대역점                  경기 성남시 수정구 성남대로 1334 (태평동, 경원프라자)
1771     금

In [26]:
import pandas as pd
import re

def refine_mega_coffee_csv():
    file_name = "mega_coffee.csv"
    
    # 1. 데이터 로드
    try:
        df = pd.read_csv(file_name)
        print(f"데이터 로드 완료! (총 {len(df)}개 매장)")
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    # --- 작업 1: 주소 컬럼 맨 오른쪽 공백 제거 ---
    # .str.rstrip()을 사용하여 오른쪽 끝의 모든 공백(스페이스, 탭, 줄바꿈 등)을 제거합니다.
    df['주소'] = df['주소'].str.rstrip()

    # --- 작업 2: 전화번호 내 한글 섞임 현상 수정 (괄호 기준 분리) ---
    def fix_address_phone_leak(row):
        address = str(row['주소'])
        phone = str(row['전화번호'])
        
        # 전화번호에 한글이 포함되어 있는지 확인
        if re.search('[가-힣]', phone):
            # 괄호 ')' 가 있다면 그 뒤를 기준으로 자름
            if ')' in phone:
                split_idx = phone.rfind(')') + 1  # 마지막 괄호 위치 찾기
                extra_address = phone[:split_idx].strip()  # 괄호까지는 주소 정보
                real_phone = phone[split_idx:].strip()    # 그 뒤가 진짜 전화번호
                
                # 기존 주소 뒤에 밀려있던 주소 정보를 붙여줌
                new_address = (address + " " + extra_address).strip()
                return pd.Series([new_address, real_phone])
        
        # 수정 사항 없으면 그대로 반환
        return pd.Series([address, phone])

    # 대상이 되는 행들에만 적용하거나 전체 적용 후 갱신
    df[['주소', '전화번호']] = df.apply(fix_address_phone_leak, axis=1)

    # --- 최종 점검 및 저장 ---
    # 다시 한번 주소의 오른쪽 공백을 정리 (정보 결합 후 생겼을 수 있는 공백 제거)
    df['주소'] = df['주소'].str.rstrip()
    
    # 수정된 데이터를 동일한 파일명으로 저장
    df.to_csv(file_name, index=False, encoding="utf-8-sig")
    
    print("-" * 50)
    print("✅ 전처리 작업 완료!")
    print(f"1. 주소 우측 공백 제거 완료")
    print(f"2. 전화번호 내 괄호 포함 주소 정보({len(df[df['전화번호'].str.contains('[가-힣]', na=False)])}건 미만 예상) 수정 완료")
    print(f"최종 데이터가 '{file_name}'에 저장되었습니다.")

if __name__ == "__main__":
    refine_mega_coffee_csv()

데이터 로드 완료! (총 4039개 매장)
--------------------------------------------------
✅ 전처리 작업 완료!
1. 주소 우측 공백 제거 완료
2. 전화번호 내 괄호 포함 주소 정보(1건 미만 예상) 수정 완료
최종 데이터가 'mega_coffee.csv'에 저장되었습니다.


In [29]:
import pandas as pd
import re

def fix_metropolitan_mobile_numbers():
    file_name = "mega_coffee.csv"
    
    # 1. 데이터 로드
    try:
        df = pd.read_csv(file_name)
        print(f"데이터 로드 완료! (현재 총 {len(df)}개 매장)")
    except Exception as e:
        print(f"파일 로드 오류: {e}")
        return

    # --- 전처리 로직 함수 ---
    def split_mobile_from_address(row):
        address = str(row['주소']).strip()
        phone = str(row['전화번호']).strip()
        
        # 대상 지역 확인 (서울, 인천, 경기)
        is_target_region = address.startswith(('서울', '인천', '경기'))
        
        # 주소 안에 '010-' 패턴이 있는지 확인
        if is_target_region and '010-' in address:
            # '010-'이 시작되는 인덱스 찾기
            match = re.search(r'010-\d{3,4}-\d{4}', address)
            
            if match:
                split_idx = match.start()
                new_address = address[:split_idx].strip()
                new_phone = address[split_idx:].strip()
                
                # 기존 전화번호 컬럼이 비어있거나 'N/A'인 경우 교체, 
                # 내용이 있다면 쉼표 등으로 구분해서 합칠 수도 있지만 보통 교체가 맞음
                return pd.Series([new_address, new_phone])
        
        # 조건에 해당하지 않으면 원래 값 유지
        return pd.Series([address, phone])

    # 2. 데이터프레임 전체에 적용
    # 'apply'를 사용하여 주소와 전화번호를 동시에 갱신합니다.
    df[['주소', '전화번호']] = df.apply(split_mobile_from_address, axis=1)

    # 3. 주소 끝에 남았을지 모르는 미세 공백 재정리
    df['주소'] = df['주소'].str.rstrip()

    # 4. 최종 저장
    df.to_csv(file_name, index=False, encoding="utf-8-sig")
    
    print("-" * 50)
    print("✅ 수도권(서울/인천/경기) 010 번호 분리 완료!")
    print(f"수정된 데이터가 '{file_name}'에 저장되었습니다.")

if __name__ == "__main__":
    fix_metropolitan_mobile_numbers()

데이터 로드 완료! (현재 총 4039개 매장)
--------------------------------------------------
✅ 수도권(서울/인천/경기) 010 번호 분리 완료!
수정된 데이터가 'mega_coffee.csv'에 저장되었습니다.


In [ ]:
import pandas as pd
import requests
import time
import os
import re
from datetime import datetime
from sqlalchemy import create_engine, text
import urllib3

# [1] SSL 인증서 경로 에러 방지 및 경고 무시
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- [설정 정보] ---
KAKAO_API_KEY = "secret"
DB_USER, DB_PASS = "root", "1234"
DB_HOST, DB_PORT, DB_NAME = "localhost", "3306", "coffee_store"
CSV_FILE = "mega_coffee.csv"

def clean_address(addr):
    """주소에서 지도 API가 인식하기 힘든 상세 정보 제거"""
    if not addr or pd.isna(addr): return ""
    # 괄호 및 내용 제거 (예: (전농동))
    addr = re.sub(r'\(.*?\)', '', addr)
    # 상세 층수 및 호수 제거 (예: 1층, 지하2층, 101호)
    addr = re.sub(r'\d+층|\d+호|지하\d+층|지상\d+층', '', addr)
    return " ".join(addr.split())

def get_kakao_coords(address):
    """카카오 API 호출"""
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}
    try:
        res = requests.get(url, headers=headers, params=params, timeout=10, verify=False)
        if res.status_code == 200:
            data = res.json()
            if data['documents']:
                return float(data['documents'][0]['y']), float(data['documents'][0]['x'])
        return None, None
    except:
        return None, None

def run_failed_data_recovery():
    if not os.path.exists(CSV_FILE):
        print(f"❌ '{CSV_FILE}' 파일이 없습니다.")
        return

    # 1. 데이터 로드
    df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    # '위도' 컬럼이 없으면 생성, 있으면 빈 값 확인
    if '위도' not in df.columns:
        df['위도'] = None
        df['경도'] = None

    # 2. 실패한 데이터만 필터링 (NaN 혹은 'None' 문자열 등)
    failed_mask = df['위도'].isna() | (df['위도'] == '') | (df['위도'].astype(str) == 'None')
    failed_indices = df[failed_mask].index

    if len(failed_indices) == 0:
        print("✅ 모든 데이터에 이미 좌표가 존재합니다.")
        return

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🛠️ 실패 데이터 {len(failed_indices)}건에 대해 카카오 API 보정을 시작합니다.")
    print("-" * 70)

    # 3. 보정 작업 수행
    for i in failed_indices:
        store_nm = df.at[i, '매장명']
        original_addr = df.at[i, '주소']
        
        # 주소 정제 후 API 호출
        refined_addr = clean_address(original_addr)
        lat, lon = get_kakao_coords(refined_addr)
        
        # 1차 실패 시 주소를 더 짧게 잘라서 재시도 (건물명 제외)
        if lat is None:
            short_addr = " ".join(refined_addr.split()[:4])
            lat, lon = get_kakao_coords(short_addr)

        if lat:
            df.at[i, '위도'] = lat
            df.at[i, '경도'] = lon
            status = "✅ 보정성공"
        else:
            status = "❌ 최종실패"

        now = datetime.now().strftime('%H:%M:%S')
        print(f"[{now}] ({i+1}/{len(df)}) {store_nm[:10]:<10} | {status} | 주소: {refined_addr[:25]}...")
        
        time.sleep(0.05)

    # 4. 결과 저장
    df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
    print("-" * 70)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ 보정 작업 완료 및 CSV 저장.")

if __name__ == "__main__":
    run_failed_data_recovery()

[00:30:30] 🛠️ 실패 데이터 4039건에 대해 카카오 API 보정을 시작합니다.
----------------------------------------------------------------------
[00:30:30] (1/4039) 강남구청역점     | ✅ 보정성공 | 주소: 서울특별시 강남구 학동로 342,...
[00:30:30] (2/4039) 강남구청점      | ✅ 보정성공 | 주소: 서울 강남구 학동로68길 7 , 상가동 지상...
[00:30:30] (3/4039) 강남보건소점     | ✅ 보정성공 | 주소: 서울 강남구 선릉로 660...
[00:30:31] (4/4039) 선릉역점       | ✅ 보정성공 | 주소: 서울 강남구 선릉로93길 26...
[00:30:31] (5/4039) 가로수길점      | ✅ 보정성공 | 주소: 서울 강남구 가로수길 19...
[00:30:31] (6/4039) 강남CGV점     | ✅ 보정성공 | 주소: 서울특별시 강남구 강남대로 438 스타플렉스...
[00:30:32] (7/4039) 강남로데오점     | ✅ 보정성공 | 주소: 서울특별시 강남구 강남대로78길 12 ,...
[00:30:32] (8/4039) 강남면허시험장점   | ✅ 보정성공 | 주소: 서울 강남구 테헤란로114길 16...
[00:30:32] (9/4039) 강남못골도서관점   | ✅ 보정성공 | 주소: 서울 강남구 자곡로 114...
[00:30:33] (10/4039) 강남세곡점      | ✅ 보정성공 | 주소: 서울특별시 강남구 헌릉로569길 13,...
[00:30:33] (11/4039) 강남역삼점      | ✅ 보정성공 | 주소: 서울 강남구 역삼로 156...
[00:30:33] (12/4039) 강남역신분당선점   | ✅ 보정성공 | 주소: 서울 강남구 강남대로 지하 396...
[00:30:33] (13/4039) 강남역점       | ✅ 보정성공 | 주소: 서울특별시 강남구 테헤

In [2]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import os

def fill_ediya_coords(file_path):
    # 1. 데이터 로드 및 위도/경도 컬럼 준비
    if not os.path.exists(file_path):
        print(f"❌ {file_path} 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")
        return

    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # '위도'와 '경도' 컬럼이 없으면 새로 생성 (빈 값으로 초기화)
    if '위도' not in df.columns:
        df['위도'] = None
    if '경도' not in df.columns:
        df['경도'] = None
    
    # 위도 또는 경도가 없는 행만 추출 (NaN 확인)
    missing_mask = df['위도'].isna() | df['경도'].isna()
    missing_df = df[missing_mask].copy()
    
    if missing_df.empty:
        print("✅ 모든 메가커피 매장의 좌표가 이미 존재합니다.")
        return df

    print(f"📡 총 {len(missing_df)}개의 빈 좌표를 발견했습니다. 수집을 시작합니다.")

    # 2. Geopy 및 RateLimiter 설정
    # Nominatim은 무료이므로 지연 시간(min_delay_seconds)을 넉넉히(1.5초 이상) 주는 것이 중요합니다.
    geolocator = Nominatim(user_agent="ediya_korea_geocoder_v1")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

    # 3. 루프 실행 및 좌표 채우기
    start_time = time.time()
    success_count = 0
    fail_count = 0

    for idx, row in missing_df.iterrows():
        # 이디야 CSV의 주소 컬럼 사용
        address = row['주소']
        try:
            # 1단계: 전체 주소로 검색 시도
            location = geocode(address)
            
            # 2단계: 실패 시 주소를 앞 3단어(시/군/구)로 잘라서 재시도 (fallback)
            if not location:
                short_addr = " ".join(address.split()[:3])
                location = geocode(short_addr)
            
            if location:
                df.at[idx, '위도'] = location.latitude
                df.at[idx, '경도'] = location.longitude
                success_count += 1
                print(f"✔️ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 위치 확보")
            else:
                fail_count += 1
                print(f"❌ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 좌표 찾기 실패")
                
        except Exception as e:
            fail_count += 1
            print(f"❗ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 오류 발생: {e}")

    # 4. 결과 저장
    # 작업 중간에 멈춰도 데이터를 잃지 않도록 덮어쓰기 방식으로 저장합니다.
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    elapsed = time.time() - start_time
    print("\n" + "="*60)
    print(f"✨ 메가커피 좌표 수집 완료! (소요 시간: {elapsed:.2f}초)")
    print(f"✅ 성공: {success_count}건 | ❌ 실패: {fail_count}건")
    print(f"📂 결과가 {file_path}에 최종 업데이트되었습니다.")
    print("="*60)

    return df

# 실행 부분
if __name__ == "__main__":
    # VS Code 프로젝트 폴더 안에 ediya.csv가 있어야 합니다.
    fill_ediya_coords('mega_coffee.csv')

📡 총 5개의 빈 좌표를 발견했습니다. 수집을 시작합니다.
✔️ [1/5] 잠실트리지움점 위치 확보
✔️ [2/5] 동탄카림3차점 위치 확보
❌ [3/5] 의왕고천파크루체점 좌표 찾기 실패
✔️ [4/5] 대전대덕브라운스톤점 위치 확보
✔️ [5/5] 부산신호점 위치 확보

✨ 메가커피 좌표 수집 완료! (소요 시간: 12.78초)
✅ 성공: 4건 | ❌ 실패: 1건
📂 결과가 mega_coffee.csv에 최종 업데이트되었습니다.
